# Modelo de Bus Regional Supply

Notebook para treinar e exportar `model.joblib` consumido por `service.py`.

In [2]:
from pathlib import Path
import importlib.util
import sys

expected_venv = (Path.cwd() / '.venv').resolve()
python_exec = Path(sys.executable).resolve()
is_expected_kernel = str(python_exec).startswith(str(expected_venv))

required_packages = ['joblib', 'numpy', 'sklearn']
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if is_expected_kernel:
    print(f'Kernel OK: {python_exec}')
else:
    print('WARNING: kernel fora do .venv do serviço.')
    print(f'Kernel atual: {python_exec}')
    print(f'Esperado   : {expected_venv / "bin" / "python"}')

if missing_packages:
    raise RuntimeError(
        'Dependências ausentes no kernel atual: '
        + ', '.join(missing_packages)
        + '. Instale no .venv e selecione esse kernel no notebook.'
    )

print('Dependências OK')

Kernel atual: /usr/bin/python3.12
Esperado   : /home/makleyston/Projects/service-composition/services/L0/bus-regional-supply/.venv/bin/python
Dependências OK


In [3]:
from __future__ import annotations

import functools
import json
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

DATASET_PATH = (Path('../../../producer/bus/dataset/tempo_real_convencional_json_070326052133.json')).resolve()
MODEL_PATH = Path('model.joblib').resolve()
TOPIC_KEY = 'service__mobility__observation__bus-telemetry'

DATASET_PATH, MODEL_PATH

(PosixPath('/home/makleyston/Projects/service-composition/producer/bus/dataset/tempo_real_convencional_json_070326052133.json'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L0/bus-regional-supply/model.joblib'))

In [4]:
rows = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
len(rows), rows[0]

(28548,
 {'EV': '105',
  'HR': '20260307171421',
  'LT': '-19.797320',
  'LG': '-44.013542',
  'NV': '31138',
  'VL': '0',
  'NL': '7979',
  'DG': '69',
  'SV': '0',
  'DT': '5400'})

In [5]:
def to_float(value):
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def hr_to_datetime(hr):
    hr_str = str(hr) if hr is not None else ''
    if len(hr_str) != 14 or not hr_str.isdigit():
        return None
    try:
        return datetime.strptime(hr_str, '%Y%m%d%H%M%S')
    except ValueError:
        return None


def group_key_for_supply(row):
    dt = hr_to_datetime(row.get('HR'))
    if dt is None:
        time_bucket = 'unknown'
    else:
        minute_bucket = (dt.minute // 10) * 10
        time_bucket = dt.replace(minute=minute_bucket, second=0).strftime('%Y%m%d%H%M')

    lt = to_float(row.get('LT'))
    lg = to_float(row.get('LG'))
    if lt is None or lg is None:
        region = 'unknown'
    else:
        region = f"{round(lt, 2)}_{round(lg, 2)}"

    return time_bucket, region


def build_labels(rows):
    vehicles_by_group = defaultdict(set)

    for row in rows:
        group = group_key_for_supply(row)
        vehicle_id = str(row.get('NV') or '')
        if vehicle_id:
            vehicles_by_group[group].add(vehicle_id)

    group_supply_count = {group: len(vehicles) for group, vehicles in vehicles_by_group.items()}
    non_zero_counts = np.array([value for value in group_supply_count.values() if value > 0], dtype=float)

    if non_zero_counts.size == 0:
        low_th, high_th = 1, 2
    else:
        q33, q66 = np.quantile(non_zero_counts, [0.33, 0.66]).tolist()
        low_th = max(1, int(round(q33)))
        high_th = max(low_th + 1, int(round(q66)))

    labels = []
    for row in rows:
        count = group_supply_count.get(group_key_for_supply(row), 0)
        if count <= low_th:
            labels.append('baixa')
        elif count <= high_th:
            labels.append('media')
        else:
            labels.append('alta')

    return labels, (low_th, high_th)

In [6]:
y, thresholds = build_labels(rows)
X_raw = [{TOPIC_KEY: row} for row in rows]

X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

pipeline = Pipeline(
    steps=[
        ('to_text', FunctionTransformer(func=functools.partial(map, json.dumps), validate=False)),
        ('vectorizer', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=2)),
        ('classifier', LogisticRegression(max_iter=800, random_state=42)),
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
thresholds

              precision    recall  f1-score   support

        alta       0.92      0.99      0.96      4725
       baixa       0.94      0.66      0.78       452
       media       0.76      0.44      0.56       533

    accuracy                           0.91      5710
   macro avg       0.87      0.70      0.76      5710
weighted avg       0.91      0.91      0.90      5710



(2, 5)

In [7]:
artifact = {
    'pipeline': pipeline,
    'target': 'regional_supply',
    'labeling_rule': {
        'grouping': 'janela de 10 minutos + celula espacial (lat/lon arredondadas em 2 casas)',
        'low_threshold': thresholds[0],
        'high_threshold': thresholds[1],
        'labels': {
            'baixa': 'count <= low_threshold',
            'media': 'low_threshold < count <= high_threshold',
            'alta': 'count > high_threshold',
        },
    },
    'dataset_path': str(DATASET_PATH),
    'dataset_size': len(rows),
}

joblib.dump(artifact, MODEL_PATH)
MODEL_PATH

PosixPath('/home/makleyston/Projects/service-composition/services/L0/bus-regional-supply/model.joblib')

In [8]:
# Sanity check no formato agregado usado pelo generic_service
sample = {TOPIC_KEY: rows[0]}
pred = pipeline.predict([sample])[0]
conf = float(pipeline.predict_proba([sample])[0].max())
pred, round(conf, 6)

(np.str_('alta'), 0.978131)